In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.data_quality_passed.emi_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA QUALITY PASSED EMI DATA
# ============================================================

source_df = spark.read.table(
    "fintech360.data_quality_passed.emi_data"
)

# ============================================================
# 2. STANDARDIZE BUSINESS COLUMNS
# ============================================================

df = (source_df
    .withColumn("business_emi_status", F.upper(F.trim(F.col("emi_status"))))
    .withColumn("business_payment_method", F.upper(F.trim(F.col("payment_method"))))
    .withColumn("business_mandate_type", F.upper(F.trim(F.col("mandate_type"))))
    .withColumn("business_emi_amount", F.col("emi_amount").cast("double"))
    .withColumn("business_paid_amount", F.col("paid_amount").cast("double"))
    .withColumn("business_late_fee", F.col("late_fee_amount").cast("double"))
    .withColumn("business_dpd", F.col("days_past_due").cast("int"))
    .withColumn("business_installment_number", F.col("installment_number").cast("int"))
    .withColumn("_due_ts", F.col("due_date").cast("long"))
    .withColumn("_paid_ts", F.col("paid_at").cast("long"))
    .withColumn("_created_ts", F.col("created_at").cast("long"))
    .withColumn("_updated_ts", F.col("updated_at").cast("long"))
)

# ============================================================
# 3. CONVERT UNIX TIMESTAMPS TO DATE/TIME DIMENSIONS
# Timestamps are assumed to be Unix epoch seconds.
# ============================================================

for c in ["due_date", "paid_at", "created_at", "updated_at"]:
    df = df.withColumn(
        f"business_{c}_datetime",
        F.to_timestamp(
            F.from_unixtime(F.col(f"_{c.split('_')[0] + '_ts'}"))
        ) if c in ["due_date", "paid_at"]
        else F.to_timestamp(
            F.from_unixtime(F.col(f"_{c.split('_')[0]}_ts"))
        )
    )

# Create date-only fields
df = (df
    .withColumn("business_due_date", F.to_date("business_due_date_datetime"))
    .withColumn("business_paid_date", F.to_date("business_paid_at_datetime"))
    .withColumn("business_due_month", F.date_format("business_due_date_datetime", "yyyy-MM"))
    .withColumn("business_due_year", F.year("business_due_date_datetime"))
    .withColumn("business_due_day", F.dayofmonth("business_due_date_datetime"))
    .withColumn("business_due_day_of_week", F.date_format("business_due_date_datetime", "EEEE"))
    .withColumn("business_paid_month", F.date_format("business_paid_at_datetime", "yyyy-MM"))
)

# ============================================================
# 4. REPAYMENT AND OUTSTANDING AMOUNT
# ============================================================

df = (df
    .withColumn(
        "emi_outstanding_amount",
        F.greatest(
            F.col("business_emi_amount") - F.col("business_paid_amount"),
            F.lit(0.0)
        )
    )
    .withColumn(
        "emi_repayment_percentage",
        F.when(
            F.col("business_emi_amount") > 0,
            F.round(
                F.col("business_paid_amount") /
                F.col("business_emi_amount") * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "emi_collection_gap",
        F.col("business_emi_amount") - F.col("business_paid_amount")
    )
    .withColumn(
        "emi_fully_paid_flag",
        F.when(
            F.col("emi_outstanding_amount") <= 0.01,
            1
        ).otherwise(0)
    )
    .withColumn(
        "emi_partially_paid_flag",
        F.when(
            (F.col("business_paid_amount") > 0) &
            (F.col("emi_outstanding_amount") > 0.01),
            1
        ).otherwise(0)
    )
    .withColumn(
        "emi_unpaid_flag",
        F.when(
            F.col("business_paid_amount") == 0,
            1
        ).otherwise(0)
    )
)

# ============================================================
# 5. EMI STATUS BUSINESS FLAGS
# ============================================================

df = (df
    .withColumn(
        "emi_paid_flag",
        F.when(F.col("business_emi_status") == "PAID", 1).otherwise(0)
    )
    .withColumn(
        "emi_overdue_flag",
        F.when(F.col("business_emi_status") == "OVERDUE", 1).otherwise(0)
    )
    .withColumn(
        "emi_pending_flag",
        F.when(F.col("business_emi_status") == "PENDING", 1).otherwise(0)
    )
    .withColumn(
        "emi_status_group",
        F.when(F.col("business_emi_status") == "PAID", "CLOSED")
         .when(F.col("business_emi_status") == "OVERDUE", "DELINQUENT")
         .when(F.col("business_emi_status") == "PENDING", "UNPAID_PENDING")
         .otherwise("OTHER_OR_UNKNOWN")
    )
)

# ============================================================
# 6. DPD BUCKETS
# ============================================================

df = df.withColumn(
    "dpd_bucket",
    F.when(F.col("business_dpd") <= 0, "CURRENT")
     .when(F.col("business_dpd") <= 30, "01_1_TO_30_DPD")
     .when(F.col("business_dpd") <= 60, "02_31_TO_60_DPD")
     .when(F.col("business_dpd") <= 90, "03_61_TO_90_DPD")
     .otherwise("04_90_PLUS_DPD")
)

# ============================================================
# 7. OVERDUE EXPOSURE
# ============================================================

df = (df
    .withColumn(
        "overdue_emi_amount",
        F.when(
            F.col("business_emi_status") == "OVERDUE",
            F.col("emi_outstanding_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "overdue_late_fee_amount",
        F.when(
            F.col("business_emi_status") == "OVERDUE",
            F.col("business_late_fee")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "total_overdue_exposure",
        F.col("overdue_emi_amount") +
        F.col("overdue_late_fee_amount")
    )
    .withColumn(
        "late_fee_to_emi_percentage",
        F.when(
            F.col("business_emi_amount") > 0,
            F.round(
                F.col("business_late_fee") /
                F.col("business_emi_amount") * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
)

# ============================================================
# 8. PAYMENT TIMELINESS
# Compare actual payment timestamp against the due timestamp.
# ============================================================

df = (df
    .withColumn(
        "payment_delay_days",
        F.when(
            F.col("_paid_ts").isNotNull() &
            F.col("_due_ts").isNotNull(),
            F.greatest(
                F.datediff(
                    F.to_date("business_paid_at_datetime"),
                    F.to_date("business_due_date_datetime")
                ),
                F.lit(0)
            )
        )
    )
    .withColumn(
        "payment_timeliness",
        F.when(
            F.col("business_emi_status") != "PAID",
            "NOT_PAID"
        ).when(
            F.col("_paid_ts").isNull(),
            "PAYMENT_DATE_MISSING"
        ).when(
            F.col("_paid_ts") <= F.col("_due_ts"),
            "PAID_ON_OR_BEFORE_DUE_DATE"
        ).otherwise("PAID_AFTER_DUE_DATE")
    )
    .withColumn(
        "paid_late_flag",
        F.when(
            (F.col("business_emi_status") == "PAID") &
            F.col("_paid_ts").isNotNull() &
            F.col("_due_ts").isNotNull() &
            (F.col("_paid_ts") > F.col("_due_ts")),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 9. INSTALLMENT SEGMENTATION
# ============================================================

df = df.withColumn(
    "installment_stage",
    F.when(F.col("business_installment_number") <= 3, "EARLY_TENURE")
     .when(F.col("business_installment_number") <= 6, "MID_EARLY_TENURE")
     .when(F.col("business_installment_number") <= 12, "MID_TENURE")
     .otherwise("LATE_TENURE")
)

# ============================================================
# 10. PAYMENT METHOD AND MANDATE CLASSIFICATION
# ============================================================

df = (df
    .withColumn(
        "payment_channel_group",
        F.when(
            F.col("business_payment_method").isin("UPI"),
            "UPI"
        ).when(
            F.col("business_payment_method").isin("CARD", "CARD_AUTOPAY"),
            "CARD"
        ).when(
            F.col("business_payment_method").isin("BANK_TRANSFER", "NACH"),
            "BANK_PAYMENT"
        ).when(
            F.col("business_payment_method") == "CASH",
            "CASH"
        ).otherwise("OTHER_OR_UNKNOWN")
    )
    .withColumn(
        "autopay_mandate_flag",
        F.when(
            F.col("business_mandate_type").isin(
                "NACH", "UPI_AUTOPAY", "CARD_AUTOPAY"
            ),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 11. BUSINESS REVIEW SIGNALS
# ============================================================

df = (df
    .withColumn(
        "high_delinquency_flag",
        F.when(F.col("business_dpd") > 90, 1).otherwise(0)
    )
    .withColumn(
        "high_late_fee_flag",
        F.when(
            F.col("business_late_fee") >= 500,
            1
        ).otherwise(0)
    )
    .withColumn(
        "repayment_review_signal",
        F.when(
            F.col("business_emi_status") == "OVERDUE",
            "OVERDUE_EMI"
        ).when(
            F.col("paid_late_flag") == 1,
            "PAID_LATE"
        ).when(
            F.col("emi_partially_paid_flag") == 1,
            "PARTIAL_REPAYMENT"
        ).when(
            F.col("emi_fully_paid_flag") == 1,
            "FULL_REPAYMENT"
        ).otherwise("PENDING_OR_OTHER")
    )
)

# ============================================================
# 12. TRANSFORMATION METADATA
# ============================================================

df = df.withColumn(
    "business_transformed_at",
    F.current_timestamp()
)

# ============================================================
# 13. FINAL BUSINESS DATAFRAME
# ============================================================

emi_data_business = df

display(emi_data_business.limit(10))


In [0]:

# Create destination schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.business_transformation
""")

# Append transformed EMI data
(emi_data_business.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.business_transformation.emi_data"))

print("EMI business transformation completed.")
print("Target: fintech360.business_transformation.emi_data")
